In [15]:
import pandas as pd
import numpy as np
from scipy import stats

In [16]:
demographic = pd.read_csv('cleaned dataset/api_data_aadhar_demographic.csv')

In [17]:
target_col = 'age between 5 and 17'  # The group driven by school admissions
date_col = 'date'
location_col = 'state'      # Change to 'district' if you want granular analysis

In [18]:
df = demographic.copy()
df[date_col] = pd.to_datetime(df[date_col])

In [19]:
# Create the Two Samples: Admission Season vs. Rest of Year
# Admission Months: June (6) and July (7)
def get_season(month):
    if month in [6, 7]:
        return 'Admission_Season'
    else:
        return 'Off_Season'

df['Season'] = df[date_col].dt.month.apply(get_season)

display(df.head())

,Unnamed: 0,date,state,district,pincode,age between 5 and 17,age 17 and above,Season
0,0,2025-03-01,uttar pradesh,Gorakhpur,273213,49,529,Off_Season
1,1,2025-03-01,andhra pradesh,Chittoor,517132,22,375,Off_Season
2,2,2025-03-01,gujarat,Rajkot,360006,65,765,Off_Season
3,3,2025-03-01,andhra pradesh,Srikakulam,532484,24,314,Off_Season
4,4,2025-03-01,rajasthan,Udaipur,313801,45,785,Off_Season


In [20]:
# Aggregate to Daily totals per Location to have enough data points for T-Test
# (T-Test needs a list of daily values, e.g., [100, 120, 105...] vs [50, 40, 55...])
daily_data = df.groupby([location_col, date_col, 'Season'])[target_col].sum().reset_index()

In [23]:
# 3. The Statistical Loop
results = []

for loc in daily_data[location_col].unique():
    subset = daily_data[daily_data[location_col] == loc]
    
    # Isolate the two samples
    season_data = subset[subset['Season'] == 'Admission_Season'][target_col]
    off_season_data = subset[subset['Season'] == 'Off_Season'][target_col]
    
    # We need at least a few data points to run a test
    if len(season_data) > 5 and len(off_season_data) > 5:
        # Perform Two-Sample T-Test (assuming unequal variance -> Welch's T-test)
        t_stat, p_val = stats.ttest_ind(season_data, off_season_data, equal_var=False)
        
        # Calculate Mean Difference (Magnitude of the spike)
        mean_season = season_data.mean()
        mean_off = off_season_data.mean()
        lift = ((mean_season - mean_off) / mean_off) * 100 if mean_off > 0 else 0
        
        # Verdict Logic
        # 1. P-value must be < 0.05 (Statistically Significant)
        # 2. Lift must be Positive (The spike must be UP, not down)
        if p_val < 0.05 and lift > 0:
            status = "Pass (Verified Spike)"
        elif p_val < 0.05 and lift < 0:
            status = "Anomaly (Significant DROP)"
        else:
            status = "FAIL (Flatline Seasonality)"
            
        results.append({
            'Location': loc,
            'Avg_Daily_Updates_Season': round(mean_season, 1),
            'Avg_Daily_Updates_OffSeason': round(mean_off, 1),
            ' seasonal_Lift_Percent': round(lift, 1),
            'P_Value': round(p_val, 5),
            'Policy_Status': status
        })

print(results)


[]


In [22]:
# 4. Create Final Report
report_df = pd.DataFrame(results)
report_df.sort_values(by='P_Value', ascending=True, inplace=True) # Strongest spikes first

dispaly(report_df.head())
        


KeyError: 'P_Value'

In [ ]:

# Export for Power BI
report_df.to_csv('school_linkage_test_results.csv', index=False)

print("Hypothesis Test Complete. 'school_linkage_test_results.csv' generated.")
print("\nTop 5 Failing States (No School Linkage Detected):")
print(report_df[report_df['Policy_Status'] == 'FAIL (Flatline Seasonality)'].head(5))